# Module 13 – Feature Engineering Pipeline

## 1. Feature Engineering Workflow

Definition: A Feature Engineering Workflow is an organized sequence of steps used to prepare raw data and convert it into useful machine learning features.

A typical workflow is:

Raw Data
    ↓
Train/Test Separation
    ↓
Feature Creation
    ↓
Numerical Transformation
    ↓
Categorical Transformation
    ↓
Feature Selection
    ↓
Final ML-Ready Features

### Hotel Booking Example

For the Hotel Bookings dataset, the workflow can include:

- Creating `total_stay_nights`
- Handling missing values
- Scaling numerical features
- Encoding categorical features
- Selecting useful features
- Applying all transformations through a reusable pipeline

The main goal is to make feature preparation consistent, reproducible, and safe from data leakage.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\HP\Sprint_6_Feature_Engineering_-_Feature_Selection\data\hotel_bookings.csv"
)

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


### Observation

The Hotel Bookings dataset is loaded as the starting point of the feature engineering workflow.

The raw dataset contains numerical and categorical features that need to be prepared before they can be used effectively by a machine learning model.

### Code Explanation

- `pd.read_csv()` loads the Hotel Bookings dataset.
- `df.head()` displays the first few records.
- This raw dataset will be passed through different feature engineering and preprocessing steps in the following sections.

### Key Point

A well-designed feature engineering workflow keeps all preprocessing steps organized and makes it easier to apply the same transformations consistently to training and unseen data.

## 2. Train/Test Separation

Definition: Train/Test Separation means dividing the dataset into two parts:

- Training data: Used to learn patterns and fit the model.
- Testing data: Used to evaluate the model on unseen data.

The test data should remain unseen during feature engineering and preprocessing.

### Hotel Booking Example

The target variable is `is_canceled`.

We separate the Hotel Booking data into:

- `X` → input features
- `y` → target variable

Then we split them into training and testing datasets.

### Why is this important?

Train/Test Separation helps provide an honest evaluation of the model and prevents information from the test data from influencing feature engineering.

In [2]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "lead_time",
    "adults",
    "children",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adr",
    "total_of_special_requests"
]

X = df[feature_cols].copy()
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (95512, 7)
Testing data: (23878, 7)


### Observation

The Hotel Booking dataset has been divided into training and testing data.

The training set contains 80% of the records, while the testing set contains 20%.

The test set will remain unseen while the feature engineering pipeline is fitted.

### Code Explanation

- `feature_cols` contains the selected input features.
- `X` stores the input features.
- `y` stores the target variable `is_canceled`.
- `train_test_split()` separates the data into training and testing sets.
- `test_size=0.2` reserves 20% of the data for testing.
- `random_state=42` makes the split reproducible.

### Key Point

The train-test split should happen before fitting preprocessing steps such as scalers, encoders, and feature-selection methods. This helps prevent data leakage.

## 3. Pipeline

Definition: A Pipeline is a sequence of data processing steps that are executed in a defined order.

A machine learning pipeline can combine multiple preprocessing operations into one reusable workflow.

### Hotel Booking Example

For the Hotel Booking dataset, a pipeline can perform steps such as:

- Handling missing values
- Scaling numerical features
- Encoding categorical features
- Preparing the final features for a machine learning model

Instead of performing each operation separately, the pipeline applies the steps consistently.

### Why is it used?

- Makes preprocessing organized and reusable.
- Ensures the same transformations are applied to training and test data.
- Reduces repeated code.
- Helps prevent data leakage.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_numeric = numeric_pipeline.fit_transform(
    X_train
)

X_test_numeric = numeric_pipeline.transform(
    X_test
)

print("Training shape:", X_train_numeric.shape)
print("Testing shape:", X_test_numeric.shape)

Training shape: (95512, 7)
Testing shape: (23878, 7)


### Observation

A reusable numerical preprocessing pipeline has been created.

The pipeline first handles missing values using the median and then standardizes the numerical features.

The pipeline is fitted only on the training data and then applied to the test data.

### Code Explanation

- `Pipeline()` combines multiple preprocessing steps.
- `SimpleImputer(strategy="median")` replaces missing numerical values with the median.
- `StandardScaler()` standardizes the numerical features.
- `fit_transform()` learns the preprocessing parameters from the training data and transforms it.
- `transform()` applies the already-learned transformations to the test data.
- This keeps preprocessing consistent and helps prevent train-test leakage.

### Key Point

A Pipeline allows multiple preprocessing operations to be treated as one reusable transformation process.

## 4. ColumnTransformer

Definition: ColumnTransformer allows different preprocessing steps to be applied to different groups of columns within the same dataset.

This is useful when a dataset contains both numerical and categorical features.

### Hotel Booking Example

The Hotel Bookings dataset contains both numerical and categorical features.

Numerical features:

- `lead_time`
- `adults`
- `children`
- `stays_in_weekend_nights`
- `stays_in_week_nights`
- `adr`
- `total_of_special_requests`

Categorical features:

- `hotel`
- `meal`
- `deposit_type`
- `customer_type`

Numerical features can be imputed and scaled, while categorical features can be imputed and one-hot encoded.

### Why is it used?

ColumnTransformer allows these different transformations to be combined into one preprocessing structure.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "lead_time",
    "adults",
    "children",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adr",
    "total_of_special_requests"
]

categorical_features = [
    "hotel",
    "meal",
    "deposit_type",
    "customer_type"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(
    X_train.assign(
        hotel=df.loc[X_train.index, "hotel"],
        meal=df.loc[X_train.index, "meal"],
        deposit_type=df.loc[X_train.index, "deposit_type"],
        customer_type=df.loc[X_train.index, "customer_type"]
    )
)

print("Processed training shape:", X_train_processed.shape)

Processed training shape: (95512, 21)


### Observation

The ColumnTransformer applies different preprocessing operations to numerical and categorical features.

Numerical features are:

1. Filled for missing values.
2. Standardized using StandardScaler.

Categorical features are:

1. Filled using the most frequent category.
2. Converted into numerical values using OneHotEncoder.

### Code Explanation

- `ColumnTransformer()` combines multiple preprocessing pipelines.
- `numeric_pipeline` handles numerical columns.
- `categorical_pipeline` handles categorical columns.
- `SimpleImputer` handles missing values.
- `StandardScaler` standardizes numerical features.
- `OneHotEncoder` converts categorical values into numerical columns.
- `handle_unknown="ignore"` prevents errors when an unseen category appears in test data.
- `fit_transform()` learns the required transformations from the training data.

### Key Point

ColumnTransformer is useful when different types of features require different preprocessing methods.

## 5. Numerical Transformations

Numerical Transformations are preprocessing operations applied to numerical features to make them suitable for machine learning models.

Common numerical transformations include:

- Missing value handling
- Standardization
- Normalization
- Log or power transformations

### Hotel Booking Example

The Hotel Bookings dataset contains numerical features such as:

- `lead_time`
- `adults`
- `children`
- `stays_in_weekend_nights`
- `stays_in_week_nights`
- `adr`
- `total_of_special_requests`

These features can have missing values and different numerical scales.

A numerical pipeline can handle missing values and standardize the features automatically.

### Why is it used?

Numerical transformations help put numerical features into a consistent form before they are given to a machine learning model.

In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_numeric = numeric_pipeline.fit_transform(
    df.loc[X_train.index, numeric_features]
)

X_test_numeric = numeric_pipeline.transform(
    df.loc[X_test.index, numeric_features]
)

print("Training shape:", X_train_numeric.shape)
print("Testing shape:", X_test_numeric.shape)

Training shape: (95512, 7)
Testing shape: (23878, 7)


### Observation

The numerical features have been processed through a reusable pipeline.

Missing values are replaced using the median, and the numerical features are then standardized.

The pipeline is fitted only on the training data and applied to the test data.

### Code Explanation

- `SimpleImputer(strategy="median")` handles missing numerical values.
- `StandardScaler()` standardizes the numerical features.
- `fit_transform()` learns the required values from the training data and transforms it.
- `transform()` applies the same learned transformation to the test data.
- This provides consistent numerical preprocessing while helping prevent data leakage.

### Key Point

Numerical transformations should be included inside the preprocessing pipeline so they can be applied consistently to both training and unseen data.

## 6. Categorical Transformations

Categorical Transformations convert categorical values into a format that machine learning models can understand.

Common categorical transformations include:

- Missing value handling
- One-Hot Encoding
- Ordinal Encoding
- Handling unknown categories

### Hotel Booking Example

The Hotel Bookings dataset contains categorical features such as:

- `hotel`
- `meal`
- `deposit_type`
- `customer_type`

These columns contain text-based categories and need to be converted into numerical representations.

One-Hot Encoding is suitable for these nominal categorical features.

### Why is it used?

Categorical transformations allow machine learning algorithms to work with categorical information without assigning an incorrect numerical order to the categories.

In [7]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

X_train_categorical = categorical_pipeline.fit_transform(
    df.loc[X_train.index, categorical_features]
)

X_test_categorical = categorical_pipeline.transform(
    df.loc[X_test.index, categorical_features]
)

print("Training shape:", X_train_categorical.shape)
print("Testing shape:", X_test_categorical.shape)

Training shape: (95512, 14)
Testing shape: (23878, 14)


### Observation

The categorical features have been processed through a reusable pipeline.

Missing categorical values are replaced with the most frequent category, and the categories are then converted into numerical columns using One-Hot Encoding.

### Code Explanation

- `SimpleImputer(strategy="most_frequent")` fills missing categorical values.
- `OneHotEncoder()` converts categorical values into numerical representations.
- `handle_unknown="ignore"` allows the pipeline to handle categories that were not present in the training data.
- `fit_transform()` learns the encoding from the training data.
- `transform()` applies the same encoding to the test data.

### Key Point

Categorical transformations should be learned from the training data and then applied to unseen data to maintain a leakage-free preprocessing workflow.

## 7. Feature Creation

Feature Creation means creating new features from existing columns to provide additional information that can be useful for machine learning.

### Hotel Booking Example

The Hotel Bookings dataset contains separate columns for:

- `stays_in_weekend_nights`
- `stays_in_week_nights`

These can be combined to create a new feature called `total_stay_nights`.

We can also combine guest-related columns to create `total_guests`.

### Why is it used?

Feature creation can represent useful business information more clearly than the original columns and can help a machine learning model identify meaningful patterns.

In [8]:
df["total_stay_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

df["total_guests"] = (
    df["adults"] +
    df["children"].fillna(0) +
    df["babies"]
)

df[
    [
        "stays_in_weekend_nights",
        "stays_in_week_nights",
        "total_stay_nights",
        "adults",
        "children",
        "babies",
        "total_guests"
    ]
].head()

,stays_in_weekend_nights,stays_in_week_nights,total_stay_nights,adults,children,babies,total_guests
0,0,0,0,2,0.0,0,2.0
1,0,0,0,2,0.0,0,2.0
2,0,1,1,1,0.0,0,1.0
3,0,1,1,1,0.0,0,1.0
4,0,2,2,2,0.0,0,2.0


### Observation

Two new features have been created from existing Hotel Booking columns:

- `total_stay_nights` represents the complete length of the guest's stay.
- `total_guests` represents the total number of guests in the booking.

These features provide a more meaningful representation of the booking information.

### Code Explanation

- `total_stay_nights` is created by adding weekend and weekday stay nights.
- `children.fillna(0)` treats missing children values as zero.
- `total_guests` combines adults, children, and babies.
- The newly created features can be included in later preprocessing and machine learning steps.

### Key Point

Feature creation should use only information that is available at prediction time and should be performed inside the appropriate pipeline when required.

## 8. Feature Selection

Feature Selection is the process of selecting the most useful features from the available feature set for a machine learning model.

Not every feature needs to be included in the final model.

### Hotel Booking Example

For predicting `is_canceled`, we can select useful booking-related features such as:

- `lead_time`
- `adults`
- `children`
- `stays_in_weekend_nights`
- `stays_in_week_nights`
- `adr`
- `total_of_special_requests`

Post-outcome features such as `reservation_status` should be excluded because they can cause feature leakage.

### Why is it used?

Feature selection can:

- Reduce unnecessary features.
- Reduce model complexity.
- Improve computational efficiency.
- Remove irrelevant or risky features.
- Help reduce the possibility of feature leakage.

In [9]:
selected_features = [
    "lead_time",
    "adults",
    "children",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adr",
    "total_of_special_requests"
]

X_selected = df[selected_features].copy()

X_selected = X_selected.fillna(0)

print("Selected features:")
print(selected_features)

print("\nNumber of selected features:", X_selected.shape[1])

Selected features:
['lead_time', 'adults', 'children', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adr', 'total_of_special_requests']

Number of selected features: 7


### Observation

The selected Hotel Booking features contain information that can reasonably be available when making a booking prediction.

Post-outcome columns such as `reservation_status` and `reservation_status_date` are not included.

### Code Explanation

- `selected_features` stores the features chosen for the model.
- `X_selected` creates a dataset containing only those features.
- `fillna(0)` handles missing values in the selected features.
- `shape[1]` gives the number of selected features.

### Key Point

Feature selection should be performed carefully and should exclude features that are irrelevant, redundant, or capable of causing data leakage.

## 9. Transformation Pipeline

Definition: A Transformation Pipeline combines multiple feature engineering and preprocessing operations into a single reusable workflow.

It ensures that the same transformations are applied consistently to training data and unseen data.

### Hotel Booking Example

For the Hotel Bookings dataset, the transformation pipeline can combine:

- Numerical missing value handling
- Numerical scaling
- Categorical missing value handling
- One-Hot Encoding

The `ColumnTransformer` can combine the numerical and categorical pipelines into one complete transformation pipeline.

### Why is it used?

- Keeps preprocessing organized.
- Makes the workflow reusable.
- Applies transformations consistently.
- Reduces repeated code.
- Helps prevent data leakage.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(
    df.loc[X_train.index, numeric_features + categorical_features]
)

X_test_processed = preprocessor.transform(
    df.loc[X_test.index, numeric_features + categorical_features]
)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (95512, 21)
Processed testing shape: (23878, 21)


### Observation

The numerical and categorical transformations have been combined into one reusable transformation pipeline.

The pipeline is fitted using only the training data and then applied to the test data.

### Code Explanation

- `numeric_pipeline` handles missing numerical values and scaling.
- `categorical_pipeline` handles missing categorical values and one-hot encoding.
- `ColumnTransformer` combines both pipelines.
- `fit_transform()` learns all required transformations from the training data.
- `transform()` applies the same learned transformations to the test data.
- `handle_unknown="ignore"` allows unseen categorical values in the test data to be handled safely.

### Key Point

A transformation pipeline makes the entire preprocessing process consistent, reusable, and less prone to accidental data leakage.

## 10. Preventing Data Leakage

Data leakage happens when information that should not be available at prediction time is used during feature engineering or model training.

A well-designed feature engineering pipeline helps prevent this by ensuring that preprocessing steps are learned only from the training data.

### Hotel Booking Example

For predicting `is_canceled`, the following practices should be followed:

- Split the data into training and testing sets first.
- Fit numerical transformations only on training data.
- Fit categorical encoders only on training data.
- Calculate aggregations only from training data.
- Perform target encoding only using training target values.
- Do not use post-outcome features such as `reservation_status`.
- Apply the fitted transformations to the test data without refitting them.

### Correct Workflow

Raw Hotel Booking Data
        ↓
Train/Test Separation
        ↓
Feature Creation
        ↓
Feature Selection
        ↓
Numerical Pipeline
        ↓
Categorical Pipeline
        ↓
ColumnTransformer
        ↓
ML-Ready Features

This approach ensures that the test data remains unseen during the learning of preprocessing parameters.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    "lead_time",
    "adults",
    "children",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adr",
    "total_of_special_requests"
]

categorical_features = [
    "hotel",
    "meal",
    "deposit_type",
    "customer_type"
]

X = df[numeric_features + categorical_features].copy()
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (95512, 21)
Testing shape: (23878, 21)


### Observation

A complete leakage-safe feature engineering pipeline has been created for the Hotel Bookings dataset.

The preprocessing steps are fitted only on the training data and then applied to the unseen test data.

### Code Explanation

- `X` contains only selected features that can be used for prediction.
- `y` contains the target variable `is_canceled`.
- `train_test_split()` separates the data before preprocessing.
- `numeric_pipeline` handles missing numerical values and scaling.
- `categorical_pipeline` handles missing categorical values and one-hot encoding.
- `ColumnTransformer` combines both pipelines.
- `fit_transform()` learns preprocessing parameters only from `X_train`.
- `transform()` applies the learned transformations to `X_test`.
- `handle_unknown="ignore"` safely handles categories that appear in the test data but were not present during training.

### Final Key Point

The most important rule is:

**Fit preprocessing on training data → Transform training data → Transform test data.**

Never allow the test data to influence the learning of preprocessing parameters.

This makes the feature engineering pipeline reusable, consistent, and less vulnerable to data leakage.